In [221]:
from src.LearnedTikhonov import LearnedTikhonov
from src.print_utils import print_kv, print_section

import torch
import h5py
import matplotlib.pyplot as plt

import numpy as np
import torch
import torch.nn as nn

from typing import Literal, get_args

plt.style.use("styles/plots.mplstyle")

In [222]:
def smoothness_regularization(model: LearnedTikhonov) -> torch.Tensor:
    """
    Penalizza variazioni brusche di R(omega) e Phi(omega).
    Assumendo che model.R e model.Phi siano nn.Parameter di shape (N_omega,).
    """
    dR   = model._get_R()[1:]   - model._get_R()[:-1]
    dPhi = model._get_phi()[1:] - model._get_phi()[:-1]
    return (dR ** 2).mean() + (dPhi ** 2).mean()

_TYPES = Literal["ReduceLR", "Cosine"]

def train(
    model: LearnedTikhonov,
    G_train: torch.Tensor,
    F_train: torch.Tensor,
    G_val:   torch.Tensor,
    F_val:   torch.Tensor,
    n_epochs: int        = 2000,
    lr: float            = 1e-3,
    weight_decay: float  = 0.0,
    checkpoint_path: str = "assets/tikhonov_learned/best_tikhonov.pt",
    scheduler_patience: int   = 50,
    scheduler_factor:   float = 0.5,
    log_every: int       = 50,
    lambda_smooth: float = 0,
    scheduler_type:_TYPES = "Cosine"
) -> dict: 

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    
    # Choose scheduler type
    options = get_args(_TYPES)
    assert scheduler_type in options, f"'{scheduler_type}' is not in {options}"
            
    if scheduler_type == "Cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=2000, T_mult=2, eta_min=1e-6
        )
    elif scheduler_type == "ReduceLR":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", patience=scheduler_patience,
            factor=scheduler_factor
        )

    loss_fn = nn.MSELoss()

    history = {"train_loss": [], "val_loss": [], "lr": []}
    best_val = float("inf")

    for epoch in range(1, n_epochs + 1):

        # ---- Training Step ----
        model.train()
        optimizer.zero_grad()
        
        pred_train = model(G_train)
        mse_train = loss_fn(pred_train, F_train)
        loss_train  = mse_train
        
        if lambda_smooth != 0:
            reg_smooth  = smoothness_regularization(model)
            loss_train  += lambda_smooth * reg_smooth
        
        loss_train.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # ---- Validation Step ----
        model.eval()
        with torch.no_grad():
            
            pred_val = model(G_val)
            mse_val = loss_fn(pred_val, F_val)
            loss_val = mse_val
            
            if lambda_smooth != 0:
                reg_smooth  = smoothness_regularization(model)
                loss_val  += lambda_smooth * reg_smooth

        train_l = loss_train.item()
        val_l   = loss_val.item()
        scheduler.step(epoch)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_l)
        history["val_loss"].append(val_l)
        history["lr"].append(current_lr)

        if val_l < best_val:
            best_val = val_l
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_loss": best_val,
            }, checkpoint_path)

        if epoch % log_every == 0 or epoch == 1:
            print(f"Epoch {epoch:5d}/{n_epochs} | "
                  f"train MSE: {train_l:.4e} | "
                  f"val MSE: {val_l:.4e} | "
                  f"lr: {current_lr:.2e}")

    print_kv(f"\nBest val MSE",f"{best_val:.4e}, {checkpoint_path}")
    return history

In [223]:
from matplotlib import figure


def plot_diagnostics(model: LearnedTikhonov, omegas: np.ndarray, tikh_w: np.ndarray = None,
                     save_path: str = "diagnostics.pdf"):
    fig, axes = plt.subplots(3, 1, figsize=(3.2, 3.6), sharex=True)

    ax1, ax2, ax3 = axes
    
    # Phi Plot
    ax1.plot(omegas, model.get_phi(), color="steelblue", lw=1.5)
    ax1.axhline(0, color="k", lw=0.2, ls="--")
    ax1.set_ylabel(r"$\phi$")
    ax1.set_title(r"Learned model $W_\theta$")
    ax1.set_ylim([-3.3, 3.3])
    ax1.grid(True, alpha=0.3)
    
    # R plot
    ax2.semilogy(omegas, model.get_R(), color="darkorange", lw=1)
    ax2.set_ylabel(r"$R$")
    ax2.set_title("")
    ax2.grid(True, alpha=0.3)

    # W plot
    ax3.semilogy(omegas, model.get_W_magnitude(), color="r", lw=1, label=r"Learned $|\mathcal{H}|$")
    
    if tikh_w is not None:
        ax3.semilogy(omegas, np.abs(tikh_w), color="b", lw=1, alpha=1, label=r"Tikhonov $|\mathcal{H}|$")
        
    ax3.set_ylabel(r"$|\mathcal{H}|$")
    ax3.set_xlabel(r"$\omega$")
    ax3.set_title("")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    plt.savefig(save_path)
    plt.close()
    print_kv(f"Diagnostic saved",f"{save_path}")
    
def plot_history(history: dict, save_path: str = "loss_history.pdf"):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots()
    
    ax.semilogy(epochs, history["train_loss"], label="Train MSE")
    ax.semilogy(epochs, history["val_loss"],   label="Val MSE", ls="--")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.set_title("Loss curve")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.savefig(save_path)
    plt.close()
    print_kv(f"Loss history saved",f"{save_path}")

def plot_prediction(model: LearnedTikhonov,
                    G_val: torch.Tensor,
                    F_val: torch.Tensor,
                    dt: float,
                    n_samples: int = 3,
                    save_path: str = "predictions.pdf",
                    save_path2: str = "predictions_mse.pdf"):

    model.eval()
    with torch.no_grad():
        f_pred = model(G_val).cpu().numpy()
    f_true = F_val.cpu().numpy()
    nt = f_true.shape[1]
    t  = np.arange(nt)

    n_show = min(n_samples, len(f_true))
    fig, axes = plt.subplots(n_show, 1, figsize=(3.2, 1.2 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]

    G_array = G_val.cpu().numpy()
    
    mse_array = np.mean((f_pred - f_true) ** 2, axis=1)
    
    mse_array_fixedtime_mean = np.mean((f_pred - f_true) ** 2, axis=0)
    mse_array_fixedtime_max = np.max((f_pred - f_true) ** 2, axis=0)
    
    for i, ax in enumerate(axes):
        
        ax.plot(t, G_array[i], label=r"$g_{obs}$", c="k")
        ax.plot(t, f_true[i], label=r"$f_{true}$", lw=3, c="b", alpha =0.5)
        ax.plot(t, f_pred[i], label=r"$f_{rec}$", c="r")
        
        mse_i = mse_array[i]
        
        mse_i_cut = np.mean((f_pred[i][:-100] - f_true[i][:-100]) ** 2)
    
        ax.text(
                0.06, 0.90,
                #f"MSE = {mse_i:.3e}\nMSEc = {mse_i_cut:.3e}",
                f"Sample MSE = {mse_i:.3e}",
                transform=ax.transAxes,
                verticalalignment='top'
            )

        #ax.axvline(x=dt*(nt-50), ls="--", c="k")
        
        ax.set_ylabel(r"$\Theta$")
        ax.grid(True, alpha=0.3)
        ax.set_ylim([-4,6])
        
        ax.minorticks_on()

    
    axes[0].legend(loc="upper right")
    axes[-1].set_xlabel("Sample")

    axes[0].set_title(f"R-phi model - MSE: {np.mean(mse_array):.3e}")
    
    print_kv("val MSE R-phi:", f"{np.mean(mse_array):.3e}")

    plt.savefig(save_path)
    plt.close()
    
    print_kv(f"Predictions saved",f"{save_path}")
    
    plt.figure(1)
    plt.grid()
    plt.plot(t, mse_array_fixedtime_mean, c="k", label=r"$MSE_t$")
    plt.plot(t, mse_array_fixedtime_max, c="r", label=r"Max $MSE_t$")
    plt.legend()
    
    plt.title(r"R-phi model $MSE_t$")
    plt.xlabel("Sample")
    plt.ylabel(r"$MSE_t$")
    plt.yscale("log")
    
    plt.savefig(save_path2)
    plt.close()
    
    print_kv(f"MSE saved",f"{save_path2}")


In [224]:
# Use this to distinguish between different datasets folder
dataset_code = "lru5_20_noise_25/"

In [225]:
# --- Load simulation parameters --------

L, Tf, nt, D, h_param, l, _, r, _ = np.loadtxt(
    f"dataset/{dataset_code}simulation_info.csv", delimiter=";", usecols=1
)
nt = int(nt)
dt = Tf / nt
n  = int(2 * nt - 1)

In [226]:
# --- Load impulse response --------------

h_vals = np.loadtxt( f"dataset/{dataset_code}impulse_response.csv", delimiter=";")
H_np   = np.fft.rfft(h_vals, n).astype(np.complex64)
nk     = len(H_np)
freqs  = np.fft.rfftfreq(n, d=dt)
omegas = freqs * 2 * np.pi
#print(f"nt={nt}, n={n}, nk={nk}, dt={dt:.4e}")

In [227]:
# ---- Load dataset ----------------------

VAL_FRACTION = 0.1
SEED         = 42
H5_PATH      =  f"dataset/{dataset_code}dataset.h5"

with h5py.File(H5_PATH, "r") as hf:
    data = hf["dataset"][:]              # (N, 2, nt)

f_all = data[:, 0, :].astype(np.float32)
g_all = data[:, 1, :].astype(np.float32)

In [228]:
# ---- Apply delay ------------------------

rolling = 0 # Simulate a delay of: tau = rolling * dt

if rolling != 0:
    g_rolled = np.roll(g_all, rolling)
    g_rolled[:,:rolling] = 0
    g_all = g_rolled
    
    print_kv("Rolling applied", rf"{rolling} elements, $\Delta_t$ = {rolling*dt}")

In [229]:
# ---- Apply noise -------------------------
noise_level = 0.25 # e.g. 0.05 = 5% of noise over the signal strenght

if noise_level > 0:
    sample_std = np.std(g_all, axis=1, keepdims=True)

    noise = np.random.normal(
        0.0,
        noise_level * sample_std,
        size=g_all.shape
    ).astype(np.float32)

    g_all += noise

    print_kv("Gaussian noise applied", f"{noise_level*100}%")

Gaussian noise applied                                           25.0%


In [230]:
# --- Validation and trainign set declaration --------

N = len(f_all)

rng       = np.random.default_rng(SEED)
idx       = rng.permutation(N)
n_val     = max(1, int(VAL_FRACTION * N))
idx_val   = idx[:n_val]
idx_train = idx[n_val:]

print_kv("Dataset", f"total={N}, train={len(idx_train)}, val={n_val}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print_kv("Device", f"{device}")

G_train = torch.from_numpy(g_all[idx_train]).to(device)
F_train = torch.from_numpy(f_all[idx_train]).to(device)
G_val   = torch.from_numpy(g_all[idx_val]).to(device)
F_val   = torch.from_numpy(f_all[idx_val]).to(device)

Dataset                                      total=1000, train=900, val=100
Device                                                            cuda


In [231]:

# --- Init model --------------------------------------------

model = LearnedTikhonov(H=H_np, n=n, nt=nt, dt=dt).to(device)
n_params = sum(p.numel() for p in model.parameters())
print_kv("Parameters number", f"(phi: {nk}, log_R: {nk})")

Parameters number                             (phi: 2000, log_R: 2000)


# 4. Training

In [232]:
perform_training = False
history = None
if perform_training:
    
    print_section("Training")
    
    # Load pretrained model
    
    load_location = f"assets/tikhonov_learned/{dataset_code}best_tikhonov.pt"
    preload_model = False
    
    if load_location != None and preload_model == True:
        ckpt = torch.load(load_location, map_location="cpu")
        model.load_state_dict(ckpt["model_state"])
        model.to(device)
        print_kv("Model preloaded", load_location)
    
    history = train(
        model        = model,
        G_train      = G_train,
        F_train      = F_train,
        G_val        = G_val,
        F_val        = F_val,
        n_epochs     = 50000,
        lr           = 1e-3,
        weight_decay = 0.0,
        checkpoint_path    = f"assets/tikhonov_learned/{dataset_code}best_tikhonov.pt",
        scheduler_patience = 50,
        scheduler_factor   = 0.5,
        log_every    = 100,
    )
    
    

# 5. Diagnostica

In [233]:
from src.Heat1DCrankNicolson import Heat1DCrankNicolson
from src.TikhonovHandler import TikhonovHandler

# Initialize solver CrankNicholson with parameters loaded from simulation_info.csv
solver = Heat1DCrankNicolson(L=L,Tf=Tf,nt=nt,D=D,h=h_param,wanted_r=r)
solver.info()

th = TikhonovHandler(solver)


-----------------------------Solver info------------------------------

Diffusion time:                                               1.00e+00
dx:                                                           8.33e-02
dt:                                                           5.00e-03
nt:                                                               2000
nx:                                                                 13
r:                                                            7.20e-01
h:                                                            1.00e+00


In [234]:
# Load trained model on CPU
ckpt = torch.load(f"assets/tikhonov_learned/{dataset_code}best_tikhonov.pt", map_location="cpu")
model.load_state_dict(ckpt["model_state"])
model.to("cpu")

# Plot R, Phi, W
plot_diagnostics(model,
                omegas,
                th._build_W(),
                save_path=f"assets/tikhonov_learned/{dataset_code}diagnostics.pdf")

# Plot loss history
if history is not None:
    plot_history(history, 
                save_path=f"assets/tikhonov_learned/{dataset_code}loss_history.pdf")

n_show = 7 # Number of example to show

# Plot prediction of subset of validation_set
plot_prediction(model, G_val.cpu(), F_val.cpu(), dt,
                n_samples=min(n_show, n_val),
                save_path=f"assets/tikhonov_learned/{dataset_code}predictions.pdf",
                save_path2=f"assets/tikhonov_learned/{dataset_code}mse.pdf")

np.savetxt("save.csv",model.get_phi(), delimiter=";")

Diagnostic saved                             assets/tikhonov_learned/lru5_20_noise_25/diagnostics.pdf
val MSE R-phi:                                               6.468e-03
Predictions saved                            assets/tikhonov_learned/lru5_20_noise_25/predictions.pdf
MSE saved                                    assets/tikhonov_learned/lru5_20_noise_25/mse.pdf


In [ ]:
G_val_np = G_val.cpu().numpy()
F_true = F_val.cpu().numpy()
F_tikh_val = np.ndarray(shape=(n_show,nt))

regularization_type_ = "closed"#"closed" 

i = 0
fig, axes = plt.subplots(n_show, 1, figsize=(3.2, 1.2 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    
    best_lambda = min([th.best_lambda(G_val_np[i]),5])
    
    F_tikh_val[i] = th.regularize(G_val_np[i], lambda_reg=best_lambda, regularization_type_=regularization_type_,)
    
    ax.plot(solver.t_array/solver.dt,  G_val_np[i], label=r"$g$", c="k")
    ax.plot(solver.t_array/solver.dt, F_true[i], label=r"$f_{true}$", lw=3, c="b", alpha =0.5)
    ax.plot(solver.t_array/solver.dt,  F_tikh_val[i], label=r"$f_{rec}$", c="r")
    
    
    mse_i = np.mean((F_tikh_val[i] - F_true[i]) ** 2)
    mse_i_cut = np.mean((F_tikh_val[i][:-100] - F_true[i][:-100]) ** 2)
    
    ax.text(
            0.06, 0.90,
            #f"MSE = {mse_i:.3e}\nMSEc = {mse_i_cut:.3e}",
            f"Sample MSE = {mse_i:.3e}",
            transform=ax.transAxes,
            verticalalignment='top'
        )
    
    ax.set_yticklabels("")
    
    #ax.axvline(x=dt*(nt-50), ls="--", c="k")
    ax.set_ylabel("")
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-4,6])
    ax.minorticks_on()

axes[-1].set_xlabel("Sample")

# Compute Tikhonov reconstruction MSE on validation set 
#F_tikh_val = torch.tensor(F_tikh_val, dtype=F_val.dtype)
#criterion = nn.MSELoss()
#loss = criterion(F_tikh_val, F_val.cpu()[:n_show])

mse_array = np.mean((F_tikh_val - F_true[:n_show]) ** 2, axis=1)
    
mse_array_fixedtime_mean = np.mean((F_tikh_val - F_true[:n_show]) ** 2, axis=0)
mse_array_fixedtime_max = np.max((F_tikh_val - F_true[:n_show]) ** 2, axis=0)
    
print_kv("val MSE tikh:", f"{np.mean(mse_array):.3e}")

axes[0].set_title(f"Tikhonov {regularization_type_} - MSE: {np.mean(mse_array):.3e} ")

plt.savefig(f"assets/tikhonov_learned/{dataset_code}predictions_tikh_{regularization_type_}.pdf")
plt.close()

plt.figure(1)
plt.grid()
plt.plot(solver.t_array/solver.dt, mse_array_fixedtime_mean, c="k", label=r"$MSE_t$")
plt.plot(solver.t_array/solver.dt, mse_array_fixedtime_max, c="r", label=r"Max $MSE_t$")
plt.legend()

plt.title(r"Tikhonov model $MSE_t$")
plt.xlabel("Sample")
plt.ylabel(r"$MSE_t$")
plt.yscale("log")

plt.savefig(f"assets/tikhonov_learned/{dataset_code}mse_tikh_{regularization_type_}.pdf")
plt.close()

print_kv(f"MSE saved",f"assets/tikhonov_learned/{dataset_code}mse_tikh_{regularization_type_}.pdf")

6.768750009458541e-06
7.609496685459884
1.3141473626117555e-06
5.790443980602483
7.609496685459884
5.790443980602483
7.609496685459884
val MSE tikh:                                                4.153e+01
MSE saved                                    assets/tikhonov_learned/lru5_20_noise_25/mse_tikh_closed.pdf
